In [ ]:
%pip install --upgrade perceptron pydantic pillow --quiet


## Configure the client
Authenticate and load the sample image.


In [ ]:
from pathlib import Path

from IPython.display import Image as IPyImage, display

from cookbook.utils import cookbook_asset
from perceptron import configure, pydantic_format, regex_format, perceive, image, text

# configure() reads PERCEPTRON_API_KEY from the environment.
configure(provider="perceptron")

SCENE_PATH = cookbook_asset("capabilities", "qna", "studio_scene.webp")
if not SCENE_PATH.exists():
    raise FileNotFoundError(f"Missing asset: {SCENE_PATH}")


## Structured output with Pydantic
Define a Pydantic model and the SDK automatically converts it to a JSON schema for constrained decoding.


In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

from dotenv import load_dotenv
load_dotenv()

class SceneAnalysis(BaseModel):
    """Structured scene analysis output."""

    scene_type: str = Field(description="Category: outdoor, indoor, urban, nature")
    main_subjects: list[str] = Field(description="Primary objects in the scene")
    mood: Literal["calm", "energetic", "dramatic", "peaceful", "tense"]
    time_of_day: Literal["morning", "afternoon", "evening", "night", "unknown"]


@perceive(model="isaac-0.2-1b", response_format=pydantic_format(SceneAnalysis))
def analyze_scene(img_path):
    return image(img_path) + text("Analyze this scene. Output in JSON with scene type, subjects, mood and time of day.")


display(IPyImage(filename=str(SCENE_PATH), width=400))
result = analyze_scene(str(SCENE_PATH))

# Parse directly into the Pydantic model
analysis = SceneAnalysis.model_validate_json(result.text)
print(f"Scene type: {analysis.scene_type}")
print(f"Subjects: {analysis.main_subjects}")
print(f"Mood: {analysis.mood}")
print(f"Time: {analysis.time_of_day}")


In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

@perceive(model="isaac-0.2-1b", response_format=regex_format("(calm|energetic|dramatic|peaceful|tense)"))
def analyze_scene(img_path):
    return image(img_path) + text("Analyze this scene, and otuput its mood.")


display(IPyImage(filename=str(SCENE_PATH), width=400))
result = analyze_scene(str(SCENE_PATH))

print(f"Mood: {result.text}")
